# Notebook 02: IMU Feature Engineering and Spectral Analysis

**Project:** Human Activity Recognition from Wearable IMU Sensor Data using Time-Series Machine Learning  
**Author:** Harish Narayanaan  
**Phase:** 2 – Feature Engineering  

### Objectives:
1. Load 9-channel raw IMU sensor windows from the UCI HAR dataset (50 Hz, 128 samples / window).
2. Compute derived Euclidean acceleration and angular velocity magnitudes.
3. Extract 13 statistical and 4 frequency-domain features across all 12 channels (204 features total).
4. Inspect feature characteristics, frequency spectra (FFT), and dynamic vs. static activity separation.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.features.feature_engineering import (
    IMUFeatureExtractor,
    extract_features,
    get_feature_names,
    ALL_CHANNEL_NAMES,
    FEATURE_TYPES,
)
from src.data.load_dataset import CHANNEL_NAMES

print("Environment initialized successfully.")

## 1. Load Processed Sensor Windows and Engineered Features

In [ ]:
data_dir = repo_root / "data" / "processed"
X_train = np.load(data_dir / "X_train.npy")
y_train = np.load(data_dir / "y_train.npy")
subj_train = np.load(data_dir / "subject_train.npy")
X_feat = np.load(data_dir / "X_train_features.npy")

print(f"Raw X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Extracted X_train_features shape: {X_feat.shape}")
print(f"Any NaNs: {np.isnan(X_feat).any()} | Any Infs: {np.isinf(X_feat).any()}")

## 2. Inspect Feature Metadata and Schema

In [ ]:
feature_names = get_feature_names()
print(f"Total features engineered: {len(feature_names)}")
print(f"Channels ({len(ALL_CHANNEL_NAMES)}): {ALL_CHANNEL_NAMES}")
print(f"Feature types per channel ({len(FEATURE_TYPES)}): {FEATURE_TYPES}")

sample_df = pd.DataFrame(X_feat[:5, :10], columns=feature_names[:10])
sample_df

## 3. Compare Dynamic (WALKING) vs Static (SITTING) Signals & Power Spectra

In [ ]:
walking_idx = np.where(y_train == 1)[0][0]
sitting_idx = np.where(y_train == 4)[0][0]

time_axis = np.arange(128) / 50.0
freq_axis = np.fft.rfftfreq(128, d=1.0 / 50.0)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time series
axes[0, 0].plot(time_axis, X_train[walking_idx, :, 0], label="body_acc_x", color="#1f77b4")
axes[0, 0].plot(time_axis, X_train[walking_idx, :, 6], label="total_acc_x", color="#ff7f0e", alpha=0.7)
axes[0, 0].set_title("WALKING: Time Series (2.56s, 50 Hz)")
axes[0, 0].set_xlabel("Time (s)")
axes[0, 0].set_ylabel("Acceleration (g)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(time_axis, X_train[sitting_idx, :, 0], label="body_acc_x", color="#1f77b4")
axes[0, 1].plot(time_axis, X_train[sitting_idx, :, 6], label="total_acc_x", color="#ff7f0e", alpha=0.7)
axes[0, 1].set_title("SITTING: Time Series (2.56s, 50 Hz)")
axes[0, 1].set_xlabel("Time (s)")
axes[0, 1].set_ylabel("Acceleration (g)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Frequency spectra
walk_psd = np.abs(np.fft.rfft(X_train[walking_idx, :, 0])) ** 2
sit_psd = np.abs(np.fft.rfft(X_train[sitting_idx, :, 0])) ** 2

axes[1, 0].plot(freq_axis, walk_psd, color="#2ca02c", lw=2)
axes[1, 0].set_title("WALKING: Power Spectral Density (body_acc_x)")
axes[1, 0].set_xlabel("Frequency (Hz)")
axes[1, 0].set_ylabel("PSD")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(freq_axis, sit_psd, color="#d62728", lw=2)
axes[1, 1].set_title("SITTING: Power Spectral Density (body_acc_x)")
axes[1, 1].set_xlabel("Frequency (Hz)")
axes[1, 1].set_ylabel("PSD")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()